# YOUR FIRST LAB — Anthropic (Claude) edition

This is the Week 1 / Day 1 lab from ed-donner's *LLM Engineering* course, **converted to use the Anthropic Claude API** instead of OpenAI, since you have an Anthropic key.

The lab itself is unchanged in spirit: we're building a tiny "web browser" that takes a URL and returns a summary — the Reader's Digest of the internet. Only the API calls have been swapped.

### What's different from the original (read this — it's the actual lesson)

| | OpenAI (original) | Anthropic (this version) |
|---|---|---|
| Import | `from openai import OpenAI` | `import anthropic` |
| Client | `OpenAI()` | `anthropic.Anthropic()` |
| Env var | `OPENAI_API_KEY` (`sk-proj-...`) | `ANTHROPIC_API_KEY` (`sk-ant-...`) |
| Call | `openai.chat.completions.create(...)` | `client.messages.create(...)` |
| System prompt | a `{"role":"system"}` message **inside** the list | a **separate** `system=` parameter |
| `max_tokens` | optional | **required** |
| Read reply | `response.choices[0].message.content` | `response.content[0].text` |

The system-prompt difference is the one that catches everyone — keep it in mind as you go.

### Kernel / setup reminder

Same as the original course: make sure you've selected the `.venv` Python kernel, and that you've run the setup so your `anthropic` package is installed (it's already in the course `requirements.txt`).

The only change to your `.env` file: it needs a line

```
ANTHROPIC_API_KEY=sk-ant-...
```

You can get a key from https://console.anthropic.com — it's under API Keys. Note Anthropic billing is separate from OpenAI; you load credits in the Anthropic console.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
import anthropic

# If you get an error running this cell, then please head over to the troubleshooting notebook!
# (Make sure `anthropic` is installed in your .venv — it's in the course requirements.txt)

# Connecting to Anthropic (Claude)

The next cell loads the environment variables from your `.env` file and checks your Anthropic key.

Anthropic keys start with `sk-ant-` (OpenAI keys started with `sk-proj-`), so the validation below checks for that instead.

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - add a line ANTHROPIC_API_KEY=sk-ant-... to your .env file")
elif not api_key.startswith("sk-ant-"):
    print("An API key was found, but it doesn't start sk-ant-; please check you're using your Anthropic key")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [ ]:
# Calling Claude is this easy. We build a list of messages, exactly like OpenAI.
# The ONE difference you'll see shortly: the system prompt is passed separately, not in this list.

message = "Hello, Claude! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user',
  'content': 'Hello, Claude! This is my first ever message to you! Hi!'}]

In [ ]:
client = anthropic.Anthropic()

# Note: max_tokens is REQUIRED by Anthropic (it's the max length of the reply).
# We use a cheap, fast model (Haiku) for these warm-up calls to keep costs tiny.

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    messages=messages,
)
response.content[0].text

"Hello! Welcome! It's great to meet you too! 👋\n\nI'm Claude, an AI assistant made by Anthropic. I'm here to help with all sorts of things—answering questions, having conversations, helping with writing, brainstorming, coding, analysis, creative projects, and much more.\n\nSince this is your first time chatting with me, feel free to ask me anything you're curious about. Is there something specific I can help you with today, or would you just like to get to know how I work?"

## OK onwards with our first project

The scraper utility (`scraper.py`) is unchanged — it has nothing to do with which LLM you use, so we just reuse it as-is.

In [ ]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
For 

## Types of prompts

Models like Claude (and GPT) have been trained to receive instructions in a particular way.

They expect:

**A system prompt** that tells them what task they're performing and what tone to use.

**A user prompt** — the conversation starter they should reply to.

⚠️ **Key Anthropic difference:** with OpenAI you put the system prompt *inside* the messages list as `{"role": "system", ...}`. With Anthropic, the system prompt is a **separate `system=` argument** to `client.messages.create(...)`, and the messages list contains only `user` (and `assistant`) turns. We'll see this in action below.

In [ ]:
# Define our system prompt - you can experiment with this later,
# e.g. change the last sentence to 'Respond in markdown in Spanish.'

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [ ]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

Anthropic's messages list looks like this — note there's **no system entry**; the system text goes in its own parameter:

```python
client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    system="system message goes here",          # <-- separate parameter
    messages=[
        {"role": "user", "content": "user message goes here"}
    ],
)
```

Compare that to OpenAI, where `{"role": "system", ...}` lived inside the `messages` list. To preview, the next 2 cells make a simple call.

In [ ]:
messages = [
    {"role": "user", "content": "What is 2 + 2?"}
]

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    system="You are a helpful assistant",
    messages=messages,
)
response.content[0].text

'2 + 2 = 4'

## And now let's build useful messages for Claude, using a function

In [ ]:
# See how this function builds ONLY the user message.
# The system prompt is passed separately when we call the API (next section).

def messages_for(website):
    return [
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
# Try this out, and then try for a few more websites

messages_for(ed)

## Time to bring it together - the Anthropic API is very simple!

Notice `system=system_prompt` sitting alongside `messages=...` — that's the whole trick.

In [ ]:
# And now: call the Anthropic API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1000,
        system=system_prompt,
        messages=messages_for(website),
    )
    return response.content[0].text

In [ ]:
summarize("https://edwarddonner.com")

In [ ]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://edwarddonner.com")

# Let's try more websites

Note this only works on websites that can be scraped with the simple `requests` approach.

JavaScript-rendered sites (React apps) won't show up — see the community-contributions folder for a Selenium version. Sites behind CloudFront may give 403 errors. But many sites work fine!

In [ ]:
display_summary("https://cnn.com")

In [ ]:
display_summary("https://anthropic.com")

## Business applications

You just called a Frontier Model's cloud API for the first time (the Anthropic one!). We applied it to **Summarization** — a classic Gen AI use case that maps onto countless business tasks: summarizing news, financial reports, resumes, support tickets. Think about where you could apply this at work, and try prototyping it.

## Before you continue — now try yourself

Use the cell below to make your own simple example. Stay with summarization for now. Idea: take the contents of an email and suggest a short subject line — the kind of feature a commercial email tool might have.

Remember the Anthropic shape: `system=...` separate, `max_tokens` required, read the reply with `response.content[0].text`.

In [ ]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list (user turn only)

messages = []  # fill this in

# Step 3: Call Claude
# response = client.messages.create(model="claude-haiku-4-5", max_tokens=1000, system=system_prompt, messages=messages)

# Step 4: print the result
# print(response.content[0].text)

## A note on models & cost

This notebook uses `claude-haiku-4-5` everywhere — Anthropic's fast, cheap model, perfect for a first lab (costs are tiny, comparable to the nano/mini models in the original).

When you want higher-quality output, swap the model string to `claude-sonnet-4-6` (stronger, a bit pricier). The rest of the code stays identical. You can confirm the current available model names at https://docs.claude.com/en/docs/about-claude/models — model strings change over time, so if a name ever errors, check there.